Cell 1 — Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_log_error

Cell 2 — Load data

In [2]:
BASE_DIR = Path(
    r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting"
)

ADVANCED_DIR = BASE_DIR / "data" / "advanced"

train = pd.read_csv(
    ADVANCED_DIR / "train_expanded_features.csv"
)

train["date"] = pd.to_datetime(
    train["date"]
)

print(train.shape)

(3000888, 50)


Cell 3 — Feature list

In [3]:
target = "sales"

features = [
    "store_nbr", "family", "onpromotion",
    "year", "month", "day", "quarter", "week", "day_of_week", "is_weekend",
    "city", "state", "type", "cluster",
    "dcoilwtico", "is_payday", "is_national_holiday", "earthquake_period",

    "sales_lag_1", "sales_lag_2", "sales_lag_3",
    "sales_lag_7", "sales_lag_14", "sales_lag_21", "sales_lag_28", "sales_lag_56",

    "rolling_mean_3", "rolling_mean_7", "rolling_mean_14",
    "rolling_mean_28", "rolling_mean_56",

    "rolling_median_7", "rolling_median_14", "rolling_median_28",

    "rolling_min_7", "rolling_min_14", "rolling_min_28",

    "rolling_max_7", "rolling_max_14", "rolling_max_28",

    "rolling_std_7", "rolling_std_14",
    "rolling_std_28", "rolling_std_56",

    "promo_rolling_mean_7",
    "promo_rolling_mean_14",
    "promo_rolling_mean_28"
]

Cell 4 — Build X and y

In [4]:
X = train[features]

y = np.log1p(
    train[target]
)

print(X.shape)
print(y.shape)

(3000888, 47)
(3000888,)


Cell 5 — Time split

In [5]:
split_date = "2017-01-01"

train_mask = train["date"] < split_date
valid_mask = train["date"] >= split_date

X_train = X.loc[train_mask]
X_valid = X.loc[valid_mask]

y_train = y.loc[train_mask]

y_valid_original = train.loc[
    valid_mask,
    target
]

print(X_train.shape)
print(X_valid.shape)

(2596374, 47)
(404514, 47)


Cell 6 — CatBoost categorical columns

In [6]:
cat_features = [
    X.columns.get_loc("store_nbr"),
    X.columns.get_loc("family"),
    X.columns.get_loc("city"),
    X.columns.get_loc("state"),
    X.columns.get_loc("type"),
    X.columns.get_loc("cluster")
]

cat_features

[0, 1, 10, 11, 12, 13]

Cell 7 — Train CatBoost

In [7]:
cat_model = CatBoostRegressor(
    iterations=1000,
    depth=8,

    learning_rate=0.05,

    loss_function="RMSE",

    eval_metric="RMSE",

    random_seed=42,

    verbose=100
)

cat_model.fit(
    X_train,
    y_train,

    cat_features=cat_features
)

print("CatBoost trained.")

0:	learn: 2.5752167	total: 2.36s	remaining: 39m 19s
100:	learn: 0.4170598	total: 3m 9s	remaining: 28m 7s
200:	learn: 0.3944071	total: 5m 55s	remaining: 23m 33s
300:	learn: 0.3851730	total: 8m 53s	remaining: 20m 38s
400:	learn: 0.3790282	total: 11m 47s	remaining: 17m 36s
500:	learn: 0.3744873	total: 14m 51s	remaining: 14m 48s
600:	learn: 0.3707515	total: 17m 43s	remaining: 11m 45s
700:	learn: 0.3680123	total: 20m 30s	remaining: 8m 44s
800:	learn: 0.3655726	total: 23m 25s	remaining: 5m 49s
900:	learn: 0.3635397	total: 26m 16s	remaining: 2m 53s
999:	learn: 0.3618998	total: 29m 5s	remaining: 0us
CatBoost trained.


Cell 8 — Validation

In [8]:
valid_log_preds = cat_model.predict(
    X_valid
)

valid_preds = np.expm1(
    valid_log_preds
)

valid_preds = np.maximum(
    valid_preds,
    0
)

rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid_original,
        valid_preds
    )
)

print("Validation RMSLE:", rmsle)

Validation RMSLE: 0.45176489933270364
